# 03 — Constraints, Examples, and Few-Shot Learning

## Scenario
Northstar must route support messages to `refund`, `shipping`, `account`, or `unknown`. Ambiguous messages must remain `unknown`; a confident but incorrect category sends users into the wrong workflow.

**Safety boundary:** This notebook uses the `google-genai` SDK and `text-embedding-004` to dynamically retrieve safe, approved examples to shape behavior.

In [ ]:
import os
import json
import random
import numpy as np
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'
EMBEDDING_MODEL = 'text-embedding-004'

class RoutingDecision(BaseModel):
    category: str = Field(description="One of: refund, shipping, account, unknown")

EVALUATION_SUITE = [
    {'id': 'clear-refund', 'message': 'I want to return my order #123.', 'expected': 'refund'},
    {'id': 'clear-shipping', 'message': 'Where is my package?', 'expected': 'shipping'},
    {'id': 'ambiguous-payment', 'message': 'Why did you charge me twice?', 'expected': 'unknown'},
    {'id': 'edge-case-return', 'message': 'Can I send back the defective item?', 'expected': 'refund'},
    {'id': 'ambiguous-account-payment', 'message': 'My account is showing a weird charge.', 'expected': 'unknown'}
]

EXAMPLE_BANK = [
    {'message': 'How do I return this?', 'category': 'refund'},
    {'message': 'I need a refund for my last purchase.', 'category': 'refund'},
    {'message': 'Track my order 456.', 'category': 'shipping'},
    {'message': 'I cannot log in.', 'category': 'account'},
    {'message': 'My credit card was double charged.', 'category': 'unknown'}, # Negative/boundary example
    {'message': 'I want to cancel my subscription.', 'category': 'account'},
    {'message': 'The delivery is late.', 'category': 'shipping'}
]

def evaluate_strategy(instruction: str, example_strategy=None):
    results = []
    total_tokens = 0
    
    for case in EVALUATION_SUITE:
        examples_text = ""
        if example_strategy:
            selected_examples = example_strategy(case['message'])
            if selected_examples:
                examples_text = "EXAMPLES:\n" + "\n".join([f"User: {ex['message']} -> {ex['category']}" for ex in selected_examples]) + "\n\n"
        
        prompt = f"{instruction}\n\n{examples_text}Message: {case['message']}"
        
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=RoutingDecision,
            )
        )
        
        output = json.loads(response.text)
        category = output.get('category', 'unknown')
            
        results.append({
            'id': case['id'],
            'expected': case['expected'],
            'observed': category
        })
        if response.usage_metadata:
            total_tokens += response.usage_metadata.total_token_count
            
    accuracy = sum(1 for r in results if r['expected'] == r['observed']) / len(results)
    return {'accuracy': accuracy, 'mean_tokens': total_tokens / len(EVALUATION_SUITE)}, results


## Baseline: Zero-Shot

We start with a direct instruction and no examples.

In [ ]:
base_instruction = "Route the support request to 'refund', 'shipping', or 'account'. If it is a payment issue or ambiguous, route to 'unknown'."

zero_shot_metrics, zero_shot_results = evaluate_strategy(base_instruction, example_strategy=None)
print("Zero-Shot Metrics:", zero_shot_metrics)
# Often, zero-shot might fail on ambiguous boundary cases like 'ambiguous-account-payment'

## Strategy 1: Static Few-Shot

We hardcode two examples into every prompt. This is predictable but uses context tokens on every request, even when the examples aren't relevant.

In [ ]:
def static_strategy(query):
    return [EXAMPLE_BANK[0], EXAMPLE_BANK[4]] # One positive (refund), one boundary (unknown payment)

static_metrics, static_results = evaluate_strategy(base_instruction, example_strategy=static_strategy)
print("Static Few-Shot Metrics:", static_metrics)

## Strategy 2: Random Few-Shot Selection

We randomly select 2 examples from the bank. This provides diversity but risks injecting irrelevant or confusing examples.

In [ ]:
def random_strategy(query):
    # Seeded for reproducibility in this notebook
    random.seed(hash(query))
    return random.sample(EXAMPLE_BANK, 2)

random_metrics, random_results = evaluate_strategy(base_instruction, example_strategy=random_strategy)
print("Random Few-Shot Metrics:", random_metrics)

## Strategy 3: Semantic Similarity Selection (RAG for Prompts)

We use `text-embedding-004` to find the 2 most semantically relevant examples from the bank for the specific user query. This maximizes relevance while minimizing context bloat.

In [ ]:
def get_embedding(text):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    )
    return response.embeddings[0].values

# Pre-compute embeddings for the bank (simulating a vector DB)
bank_embeddings = [np.array(get_embedding(ex['message'])) for ex in EXAMPLE_BANK]

def similarity_strategy(query):
    query_emb = np.array(get_embedding(query))
    
    # Compute cosine similarity
    similarities = []
    for bank_emb in bank_embeddings:
        sim = np.dot(query_emb, bank_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(bank_emb))
        similarities.append(sim)
        
    # Get top 2 indices
    top_indices = np.argsort(similarities)[-2:][::-1]
    return [EXAMPLE_BANK[i] for i in top_indices]

sim_metrics, sim_results = evaluate_strategy(base_instruction, example_strategy=similarity_strategy)
print("Semantic Similarity Few-Shot Metrics:", sim_metrics)

## Conclusion: Accuracy vs. Context Cost

By measuring both accuracy and context size (tokens), we can make an informed decision about our few-shot strategy.

In [ ]:
print(f"Zero-Shot:  Accuracy {zero_shot_metrics['accuracy']:.2f} | Tokens/Req {zero_shot_metrics['mean_tokens']:.1f}")
print(f"Static:     Accuracy {static_metrics['accuracy']:.2f} | Tokens/Req {static_metrics['mean_tokens']:.1f}")
print(f"Random:     Accuracy {random_metrics['accuracy']:.2f} | Tokens/Req {random_metrics['mean_tokens']:.1f}")
print(f"Similarity: Accuracy {sim_metrics['accuracy']:.2f} | Tokens/Req {sim_metrics['mean_tokens']:.1f}")

print("\nSemantic similarity ensures we only spend tokens on highly relevant boundary or positive examples, avoiding the randomness of naive selection.")